In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from openpyxl.utils import indexed_list


#1st way: more control and can avoid error when loading bid dataset
import sql libary
con = python.connect(localhost, password,...)
cursor = con.sursor()
cursor.Execute('Select...')
cursor.Ferdi...

#2nd way

import panda 
read = pd.from_sql or read sql (table name, )

In [2]:
# Load the user_review dataset
#user_review = pd.read_excel(r'/Users/huyennguyen/Documents/MDD/SQL/Data Files/UserReviewsClean43LIWC.xlsx')
#user_review.head()

In [3]:
#user_review.columns

In [4]:
#user_review['url'].unique()

In [5]:

# Load the expert_review dataset
#expert_review = pd.read_excel(r'/Users/huyennguyen/Documents/MDD/SQL/Data Files/ExpertReviewsClean43LIWC.xlsx')
#expert_review.head()

In [6]:
#expert_review.columns

In [7]:
#user_review.info()

Change Data type and remove quotation marks

In [8]:
#expert_review.info()

### Sales data need to be cleaned:
- Unnamed column: remove
- Release Date formate


In [9]:
#sales = pd.read_excel(r'/Users/huyennguyen/Documents/MDD/SQL/Data Files/sales.xlsx')
#sales.head()

In [10]:
#sales['creative_type'].unique()

In [11]:
#sales['url'].value_counts()

In [12]:
#sales.info()

### Cleaning MetaData
1. Irrelevant data
- All columns look purely irrelevant 

2. Duplicates
- No fully duplicated rows

3. Data type conversion
- title, studio, rating, director: object change to -> string
- runtime: float64 - OK
- userscore: float64 - OK
- RelDate: datetime64 - OK

4. Syntax errors
- title, studio, cast, director: need to be encoding, decode as Latin‑1
- title: lowercase each word, strip whitespace
- rating: 
    - strip("|") and ("-")

5. Missing values: Keeping all as NaN

6. Normalization and Scalling data
- cast, genre, awards: Standardize the spacing before splitting into separate rows.
- cast, genre, awards: splitting into junction tables: movie_cast, movie_genre, movie_awards
- studio: normalizing name order for consistent grouping "Weinstein Company, The" -> "The Weinstein Company"

In [2]:
meta = pd.read_excel(r'/Users/huyennguyen/Documents/MDD/SQL/Data Files/metaClean43Brightspace.xlsx')
meta.head()

,url,title,studio,rating,runtime,cast,director,genre,summary,awards,metascore,userscore,RelDate
0,https://www.metacritic.com/movie/!women-art-re...,!Women Art Revolution,Hotwire Productions,| Not Rated,83.0,NaN,Lynn Hershman-Leeson,Documentary,NaN,NaN,70,NaN,2011-06-01
1,https://www.metacritic.com/movie/10-cloverfiel...,10 Cloverfield Lane,Paramount Pictures,| PG-13,104.0,"John Gallagher Jr.,John Goodman,Mary Elizabeth...",Dan Trachtenberg,"Action,Sci-Fi,Drama,Mystery,Thriller,Horror","Waking up from a car accident, a young woman (...","#18MostDiscussedMovieof2016 , #1MostSharedMovi...",76,7.7,2016-03-11
2,https://www.metacritic.com/movie/10-items-or-less,10 Items or Less,Click Star,| R,82.0,"Jonah Hill,Morgan Freeman,Paz Vega",Brad Silberling,"Drama,Comedy,Romance",While researching a role as a supermarket mana...,NaN,54,5.8,2006-12-01
3,https://www.metacritic.com/movie/10-years,10 Years,Anchor Bay Entertainment,| R,100.0,"Channing Tatum,Chris Pratt,Jenna Dewan",Jamie Linden,"Drama,Comedy,Romance",NaN,NaN,61,6.9,2012-09-14
4,https://www.metacritic.com/movie/100-bloody-acres,100 Bloody Acres,Music Box Films,| Not Rated,91.0,NaN,Cameron Cairnes,"Horror,Comedy",Reg and Lindsay run an organic fertilizer busi...,NaN,63,7.5,2013-06-28


Checking data type

In [3]:
print(meta.info()) # Check data type

print(meta.describe()) # Check statistic data values

print(meta.shape) # Check numbers of rows and columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11364 entries, 0 to 11363
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   url        11364 non-null  object        
 1   title      11364 non-null  object        
 2   studio     11014 non-null  object        
 3   rating     10297 non-null  object        
 4   runtime    11109 non-null  float64       
 5   cast       7662 non-null   object        
 6   director   11350 non-null  object        
 7   genre      11344 non-null  object        
 8   summary    5467 non-null   object        
 9   awards     4387 non-null   object        
 10  metascore  11364 non-null  int64         
 11  userscore  9259 non-null   float64       
 12  RelDate    11364 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(2), int64(1), object(9)
memory usage: 1.1+ MB
None
(11364, 13)
            runtime     metascore    userscore                        RelDate
count  111

#### Shape
11,364 rows × 13 columns 

#### Descriptive Statistic
- Coverage spans movies released 2000–2021, median release mid-2012.
- metascore is complete, no missing values and spans its full theoretical 1–100 range
- userscore (0–10 scale) is missing for ~18.5% of titles — user scores are less consistently captured than critic scores.
- runtime max of 808 minutes is an outlier worth flagging,likely a mini-series or data entry issue. This requires validation before use in analysis.
- summary (51.9%) and awards (61.4%) are the two columns with the heaviest missingness. usable for enrichment but not for any completeness, dependent analysis.
- cast missing in ~32.6% of rows
- cast/genre/awards needed splitting into junction tables


In [15]:
#check duplicated values, setting keep on False, all duplicates are True.
meta.duplicated(keep=False).value_counts


<bound method IndexOpsMixin.value_counts of 0        False
1        False
2        False
3        False
4        False
         ...  
11359    False
11360    False
11361    False
11362    False
11363    False
Length: 11364, dtype: bool>

There is no any duplicated value

In [ ]:
# check the missing values inside each column
for col in meta.columns:
    print(f'\n-- {col} --')
    print(meta[col].isnull().value_counts()) # count missing value

    missing_pct = meta[col].isnull().mean() * 100 # Calculate missing percentage

    print(f"Missing %: {missing_pct:.2f}%")


-- url --
url
False    11364
Name: count, dtype: int64
Missing %: 0.00%

-- title --
title
False    11364
Name: count, dtype: int64
Missing %: 0.00%

-- studio --
studio
False    11014
True       350
Name: count, dtype: int64
Missing %: 3.08%

-- rating --
rating
False    10297
True      1067
Name: count, dtype: int64
Missing %: 9.39%

-- runtime --
runtime
False    11109
True       255
Name: count, dtype: int64
Missing %: 2.24%

-- cast --
cast
False    7662
True     3702
Name: count, dtype: int64
Missing %: 32.58%

-- director --
director
False    11350
True        14
Name: count, dtype: int64
Missing %: 0.12%

-- genre --
genre
False    11344
True        20
Name: count, dtype: int64
Missing %: 0.18%

-- summary --
summary
True     5897
False    5467
Name: count, dtype: int64
Missing %: 51.89%

-- awards --
awards
True     6977
False    4387
Name: count, dtype: int64
Missing %: 61.40%

-- metascore --
metascore
False    11364
Name: count, dtype: int64
Missing %: 0.00%

-- userscore 

- Flag missing values as NaN

In [17]:

# the unique values inside each column
for col in meta.columns:
    print(f'\n-- {col} --')
    print(meta[col].unique())


-- url --
['https://www.metacritic.com/movie/!women-art-revolution'
 'https://www.metacritic.com/movie/10-cloverfield-lane'
 'https://www.metacritic.com/movie/10-items-or-less' ...
 'https://www.metacritic.com/movie/zoom-2016'
 'https://www.metacritic.com/movie/zootopia'
 'https://www.metacritic.com/movie/zus-zo']

-- title --
['!Women Art Revolution' '10 Cloverfield Lane' '10 Items or Less' ...
 'Zoom' 'Zootopia' 'Zus & zo']

-- studio --
['Hotwire Productions' 'Paramount Pictures' 'Click Star' ...
 'Film Desk, The' 'myCinema' '20th Century Fox International Classics']

-- rating --
['| Not Rated' '| PG-13' '| R' nan '| G' '| TV-MA' '| Unrated' '| PG'
 '| NR' '| NC-17' '| TV-14' '| MA-17' '| TV-PG' '| PG--13' '| TV-G'
 '| Open' '| Approved' '| M' '| PG-13`' '| M/PG']

-- runtime --
[ 83. 104.  82. 100.  91.  93. 117. 109.  99.  88. 107. 110.  81. 159.
 105.  87. 118.  76. 108. 130. 134.  89.  94.  97. 141.  98. 144. 114.
 120. 102.  96.  86. 125. 119.  79.  85.  90. 112. 158. 129. 12

## Data Type Conversion

In [34]:
# Convert object columns to string
meta['title'] = meta['title'].astype('string')
meta['studio'] = meta['studio'].astype('string')
meta['rating'] = meta['rating'].astype('string')
meta['director'] = meta['director'].astype('string')

meta.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11364 entries, 0 to 11363
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   url          11364 non-null  object        
 1   title        11364 non-null  string        
 2   studio       11014 non-null  string        
 3   rating       10297 non-null  string        
 4   runtime      11109 non-null  float64       
 5   cast         7662 non-null   object        
 6   director     11350 non-null  string        
 7   genre        11344 non-null  object        
 8   summary      5467 non-null   object        
 9   awards       4387 non-null   object        
 10  metascore    11364 non-null  int64         
 11  userscore    9259 non-null   float64       
 12  RelDate      11364 non-null  datetime64[ns]
 13  cast_list    7662 non-null   object        
 14  genre_list   11344 non-null  object        
 15  awards_list  4387 non-null   object        
dtypes: d

## Cleaning Syntax Errors

### Encoding and Decoding texts
- title, studio, cast, director: need to be encoding, decode as Latin‑1

In [19]:
def try_encoding(text): # define function to receive one text. e.g: 'GÃ©rard'
    if pd.isna(text):   # if value is NaN -> return NaN
        return text
    try:
        return text.encode('latin1').decode('utf-8')
    except (UnicodeDecodeError, UnicodeEncodeError):
        return text  # leave text unchanged if it is normal

for col in ['title', 'studio', 'cast', 'director', 'summary']: #looping columns that texts need to be enconded
    meta[col] = meta[col].apply(try_encoding)                  #apply encoding function and run on every value in these columns

# check if the encoding correction worked
meta.loc[meta['cast'].str.contains('GÃ©rard', na=False), 'cast'] # find title column for value contains 'GÃ©rard', missing values treat as False,  meta.loc[condition, 'cast']: Select the rows where the condition is true, and show only the cast column.

# "GÃ©rard" -> Latin-1 encoding -> BYTES: 47 C3 A9 72 61 72 64 (C3 A9 = the original UTF-8 bytes for é) -> decode('itf-8') -> correct text: Gérard

Series([], Name: cast, dtype: object)

In [4]:
# Strip whitespace ['title', 'studio', 'rating', 'cast', 'director', 'genre', 'summary', 'awards']

text_cols = ['title', 'studio', 'rating', 'cast', 'director', 'genre', 'summary', 'awards']
for col in text_cols:
    meta[col] = meta[col].str.strip()

In [ ]:
# lowercase title characters, and replace symbols to normal whitespace

meta['title'] = meta['title'].str.lower()
meta['title'] = meta['title'].str.replace('&', 'and', regex=True)
meta['title'] = meta['title'].str.replace(r'[^\w\s]', '', regex=True)

meta['title'].value_counts()


    

title
eden                    3
lucky                   3
innocence               3
the king                3
the outsider            3
                       ..
x games 3d the movie    1
yosemite                1
yes god yes             1
year by the sea         1
zus and zo              1
Name: count, Length: 11139, dtype: Int64

- Remove (|,-,..) in rating column

In [21]:
# Remove symbols such as !, ?, ,, ., etc. 
meta['rating'] = meta['rating'].str.replace(r'[^\w\s]', '', regex=True)

print(meta['rating'].value_counts(dropna=False))


rating
 R            3515
 Not Rated    2960
 PG13         2031
<NA>          1067
 PG            816
 Unrated       384
 TVMA          200
 NR            128
 G             124
 TV14           56
 NC17           40
 TVPG           23
 TVG             7
 Open            5
 Approved        4
 M               2
 MA17            1
 MPG             1
Name: count, dtype: Int64


In [23]:
# Split multi-valued columns (cast, genre, awards) into lists
#    so each can later be exploded into its own junction table
#    (movie_cast, movie_genre, movie_awards) per your assignment notes
# ============================================================
meta['cast_list']   = meta['cast'].str.split(',')
meta['genre_list']  = meta['genre'].str.split(',')
meta['awards_list'] = meta['awards'].str.split(r'\s*,\s*', regex=True)  # awards had inconsistent spacing



In [24]:
# strip whitespace within each list item
meta['cast_list']   = meta['cast_list'].apply(lambda lst: [x.strip() for x in lst] if isinstance(lst, list) else lst)
meta['genre_list']  = meta['genre_list'].apply(lambda lst: [x.strip() for x in lst] if isinstance(lst, list) else lst)
meta['awards_list'] = meta['awards_list'].apply(lambda lst: [x.strip() for x in lst] if isinstance(lst, list) else lst)


In [25]:
# build a normalized long-format movie with genre table
movie_genre = (
    meta[['title', 'genre_list']]
    .explode('genre_list')
    .rename(columns={'genre_list': 'genre'})
    .dropna(subset=['genre'])
    .reset_index(drop=True)
)
movie_genre.head(10)


,title,genre
0,!Women Art Revolution,Documentary
1,10 Cloverfield Lane,Action
2,10 Cloverfield Lane,Sci-Fi
3,10 Cloverfield Lane,Drama
4,10 Cloverfield Lane,Mystery
5,10 Cloverfield Lane,Thriller
6,10 Cloverfield Lane,Horror
7,10 Items or Less,Drama
8,10 Items or Less,Comedy
9,10 Items or Less,Romance


In [26]:
# build a normalized long-format movie with cast table
movie_cast = (
    meta[['title', 'cast_list']]
    .explode('cast_list')
    .rename(columns={'cast_list': 'cast'})
    .dropna(subset=['cast'])
    .reset_index(drop=True)
)
movie_cast.head(10)

,title,cast
0,10 Cloverfield Lane,John Gallagher Jr.
1,10 Cloverfield Lane,John Goodman
2,10 Cloverfield Lane,Mary Elizabeth Winstead
3,10 Items or Less,Jonah Hill
4,10 Items or Less,Morgan Freeman
5,10 Items or Less,Paz Vega
6,10 Years,Channing Tatum
7,10 Years,Chris Pratt
8,10 Years,Jenna Dewan
9,"10,000 BC",Camilla Belle


In [27]:
# build a normalized long-format movie with awards table
movie_awards = (
    meta[['title', 'awards_list']]
    .explode('awards_list')
    .rename(columns={'awards_list': 'awards'})
    .dropna(subset=['awards'])
    .reset_index(drop=True)
)
movie_awards.head(10)

,title,awards
0,10 Cloverfield Lane,#18MostDiscussedMovieof2016
1,10 Cloverfield Lane,#1MostSharedMovieof2016
2,"10,000 BC",#23MostDiscussedMovieof2008
3,"10,000 BC",#27MostSharedMovieof2008
4,102 Dalmatians,#73MostDiscussedMovieof2000
5,12,#97BestMovieof2009
6,12,#18MostSharedMovieof2009
7,12 Rounds,#71MostSharedMovieof2009
8,12 Strong,#77MostDiscussedMovieof2018
9,12 Strong,#21MostSharedMovieof2018


In [ ]:
# build a normalized long-format movie with the whole table
movie_cleaned = (
    meta[['url','title','studio', 'rating', 'runtime', 'cast_list',	'director', 'awards_list', 'genre_list', 'metascore', 'userscore', 'RelDate', 'summary']]
    .explode('awards_list')
    .explode( 'cast_list')
    .explode('genre_list')
    .rename(columns={'awards_list': 'awards'})
    .rename(columns={'cast_list': 'cast'})
    .rename(columns={'genre_list': 'genre'})
    .dropna(subset=['awards', 'cast', 'genre'])
    .reset_index(drop=True)
)
movie_cleaned.head(10)


,url,title,studio,rating,runtime,cast,director,awards,genre,metascore,userscore,RelDate,summary
0,https://www.metacritic.com/movie/10-cloverfiel...,10 cloverfield lane,Paramount Pictures,PG13,104.0,John Gallagher Jr.,Dan Trachtenberg,#18MostDiscussedMovieof2016,Action,76,7.7,2016-03-11,"Waking up from a car accident, a young woman (..."
1,https://www.metacritic.com/movie/10-cloverfiel...,10 cloverfield lane,Paramount Pictures,PG13,104.0,John Gallagher Jr.,Dan Trachtenberg,#18MostDiscussedMovieof2016,Sci-Fi,76,7.7,2016-03-11,"Waking up from a car accident, a young woman (..."
2,https://www.metacritic.com/movie/10-cloverfiel...,10 cloverfield lane,Paramount Pictures,PG13,104.0,John Gallagher Jr.,Dan Trachtenberg,#18MostDiscussedMovieof2016,Drama,76,7.7,2016-03-11,"Waking up from a car accident, a young woman (..."
3,https://www.metacritic.com/movie/10-cloverfiel...,10 cloverfield lane,Paramount Pictures,PG13,104.0,John Gallagher Jr.,Dan Trachtenberg,#18MostDiscussedMovieof2016,Mystery,76,7.7,2016-03-11,"Waking up from a car accident, a young woman (..."
4,https://www.metacritic.com/movie/10-cloverfiel...,10 cloverfield lane,Paramount Pictures,PG13,104.0,John Gallagher Jr.,Dan Trachtenberg,#18MostDiscussedMovieof2016,Thriller,76,7.7,2016-03-11,"Waking up from a car accident, a young woman (..."
5,https://www.metacritic.com/movie/10-cloverfiel...,10 cloverfield lane,Paramount Pictures,PG13,104.0,John Gallagher Jr.,Dan Trachtenberg,#18MostDiscussedMovieof2016,Horror,76,7.7,2016-03-11,"Waking up from a car accident, a young woman (..."
6,https://www.metacritic.com/movie/10-cloverfiel...,10 cloverfield lane,Paramount Pictures,PG13,104.0,John Goodman,Dan Trachtenberg,#18MostDiscussedMovieof2016,Action,76,7.7,2016-03-11,"Waking up from a car accident, a young woman (..."
7,https://www.metacritic.com/movie/10-cloverfiel...,10 cloverfield lane,Paramount Pictures,PG13,104.0,John Goodman,Dan Trachtenberg,#18MostDiscussedMovieof2016,Sci-Fi,76,7.7,2016-03-11,"Waking up from a car accident, a young woman (..."
8,https://www.metacritic.com/movie/10-cloverfiel...,10 cloverfield lane,Paramount Pictures,PG13,104.0,John Goodman,Dan Trachtenberg,#18MostDiscussedMovieof2016,Drama,76,7.7,2016-03-11,"Waking up from a car accident, a young woman (..."
9,https://www.metacritic.com/movie/10-cloverfiel...,10 cloverfield lane,Paramount Pictures,PG13,104.0,John Goodman,Dan Trachtenberg,#18MostDiscussedMovieof2016,Mystery,76,7.7,2016-03-11,"Waking up from a car accident, a young woman (..."


In [ ]:
movie_cleaned.shape

In [ ]:
meta_cleaned.to_excel("meta_cleaned_v1.xlsx", index=False)

print("Saved successfully")
print(meta_cleaned.shape)

In [ ]:
import os
print(os.getcwd())